In [1]:
from pathlib import Path
import math

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules


# =========================================================
# 1. 경로 설정
# =========================================================

BASE_DIR = Path(r"D:\OneDrive\바탕 화면\Yoonity_KAIA")
DATA_DIR = BASE_DIR / "outputs"
RESULT_DIR = DATA_DIR / "KAIA_DA_k6_model_b_association_rules"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "KAIA_DA_k=6_model_b_input.csv"
LOOKUP_PATH = DATA_DIR / "KAIA_DA_k=6_model_b_case_lookup.csv"
CLUSTER_PATH = DATA_DIR / "KAIA_DA_k=6_model_b_cluster_result_asw.csv"


# =========================================================
# 2. 분석 설정
# =========================================================

# 최종 Model B CCA 입력변수
VARIABLES = [
    "철도구분",
    "열차종류",
    "지연운행 현황",
    "지연운행 원인",
    "근본원인별 원인",
    "장소유형A",
]

# 전달받은 코드의 기준
# 논문에 사용하기 전 선정 근거를 확인해야 함
MIN_SUPPORT = 0.20
MIN_CONFIDENCE = 0.60
MIN_LIFT = 1.20

# 최대 항목 수:
# antecedent와 consequent를 합하여 최대 3개 항목
MAX_LEN = 3

CLUSTER_NAMES = {
    1: "여객열차·차량설비 고장형",
    2: "도시전동차·복합설비 장애형",
    3: "외부환경 요인형",
    4: "신호제어설비 장애형",
    5: "무정차통과·운전취급 부주의형",
    6: "전철전력설비·급전장애형",
}

EXPECTED_CLUSTER_COUNTS = {
    1: 337,
    2: 132,
    3: 97,
    4: 90,
    5: 48,
    6: 35,
}

EXPECTED_TOTAL_N = 739


# =========================================================
# 3. 최종 저장 결과 불러오기
# =========================================================

for path in [INPUT_PATH, LOOKUP_PATH, CLUSTER_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"필수 파일을 찾을 수 없습니다: {path}")

model_input_df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
case_lookup_df = pd.read_csv(LOOKUP_PATH, encoding="utf-8-sig")
cluster_df = pd.read_csv(CLUSTER_PATH, encoding="utf-8-sig")


# =========================================================
# 4. 식별자 및 군집 결과 검증
# =========================================================

for name, frame in {
    "model_input": model_input_df,
    "case_lookup": case_lookup_df,
    "cluster_result": cluster_df,
}.items():

    if "case_id" not in frame.columns:
        raise KeyError(f"{name}에 case_id가 없습니다.")

    if frame["case_id"].isna().any():
        raise ValueError(f"{name}의 case_id에 결측치가 있습니다.")

    if not frame["case_id"].is_unique:
        duplicated = frame.loc[
            frame["case_id"].duplicated(keep=False),
            "case_id"
        ].tolist()

        raise ValueError(
            f"{name}의 case_id가 중복되었습니다: {duplicated[:10]}"
        )

if len(model_input_df) != EXPECTED_TOTAL_N:
    raise ValueError(
        f"Model B 입력 관측치 수 불일치: "
        f"{len(model_input_df)} != {EXPECTED_TOTAL_N}"
    )

if len(case_lookup_df) != EXPECTED_TOTAL_N:
    raise ValueError(
        f"case lookup 관측치 수 불일치: "
        f"{len(case_lookup_df)} != {EXPECTED_TOTAL_N}"
    )

if len(cluster_df) != EXPECTED_TOTAL_N:
    raise ValueError(
        f"군집 결과 관측치 수 불일치: "
        f"{len(cluster_df)} != {EXPECTED_TOTAL_N}"
    )

actual_clusters = set(
    pd.to_numeric(cluster_df["cluster"], errors="raise")
    .astype(int)
    .unique()
)

if actual_clusters != {1, 2, 3, 4, 5, 6}:
    raise ValueError(
        f"군집 번호가 1~6과 일치하지 않습니다: {actual_clusters}"
    )

cluster_df["cluster"] = cluster_df["cluster"].astype(int)

actual_cluster_counts = (
    cluster_df["cluster"]
    .value_counts()
    .sort_index()
    .to_dict()
)

if actual_cluster_counts != EXPECTED_CLUSTER_COUNTS:
    raise ValueError(
        "기존 KAIA_DA_k=6 군집 규모와 일치하지 않습니다.\n"
        f"기대값: {EXPECTED_CLUSTER_COUNTS}\n"
        f"실제값: {actual_cluster_counts}"
    )


# =========================================================
# 5. case_id 기반으로 연결
# =========================================================

analysis_df = (
    model_input_df
    .merge(
        cluster_df[["case_id", "cluster"]],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(analysis_df) != EXPECTED_TOTAL_N:
    raise ValueError(
        f"병합 후 관측치 수 불일치: "
        f"{len(analysis_df)} != {EXPECTED_TOTAL_N}"
    )

input_case_ids = set(model_input_df["case_id"])
cluster_case_ids = set(cluster_df["case_id"])

if input_case_ids != cluster_case_ids:
    only_input = sorted(input_case_ids - cluster_case_ids)
    only_cluster = sorted(cluster_case_ids - input_case_ids)

    raise ValueError(
        "Model B 입력과 군집 결과의 case_id 집합이 다릅니다.\n"
        f"입력에만 존재: {only_input[:10]}\n"
        f"군집 결과에만 존재: {only_cluster[:10]}"
    )


# =========================================================
# 6. 변수 검증
# =========================================================

missing_columns = [
    col for col in VARIABLES
    if col not in analysis_df.columns
]

if missing_columns:
    raise KeyError(
        f"Model B 입력 파일에 필요한 변수가 없습니다: {missing_columns}"
    )

missing_counts = analysis_df[VARIABLES].isna().sum()

if missing_counts.gt(0).any():
    raise ValueError(
        "연관규칙 변수에 결측치가 있습니다.\n"
        f"{missing_counts[missing_counts.gt(0)]}"
    )


# =========================================================
# 7. 트랜잭션 행렬 생성 함수
# =========================================================

def make_transaction_matrix(frame, variables):
    """
    각 관측치를 하나의 transaction으로 처리하고,
    '변수명=범주명' 형식으로 이진 행렬을 생성한다.

    변수명을 포함하는 이유:
    서로 다른 변수에서 동일한 범주명이 사용되는 경우를 구분하기 위함.
    """

    categorical = frame[variables].copy()

    for col in variables:
        categorical[col] = categorical[col].astype(str).str.strip()

    matrix = pd.get_dummies(
        categorical,
        columns=variables,
        prefix=variables,
        prefix_sep="=",
        dtype=bool,
    )

    expected_items_per_row = len(variables)
    actual_items_per_row = matrix.sum(axis=1)

    if not actual_items_per_row.eq(expected_items_per_row).all():
        raise ValueError(
            "일부 관측치의 항목 수가 변수 수와 일치하지 않습니다."
        )

    return matrix


def itemset_to_text(itemset):
    return " & ".join(sorted(str(item) for item in itemset))


def get_variable_names(itemset):
    return {
        str(item).split("=", 1)[0]
        for item in itemset
    }


# =========================================================
# 8. 군집별 연관규칙 계산
# =========================================================

all_rules = []
summary_rows = []

for cluster_id in range(1, 7):

    subset = analysis_df.loc[
        analysis_df["cluster"] == cluster_id
    ].copy()

    expected_n = EXPECTED_CLUSTER_COUNTS[cluster_id]

    if len(subset) != expected_n:
        raise ValueError(
            f"군집 {cluster_id} 관측치 수 불일치: "
            f"{len(subset)} != {expected_n}"
        )

    matrix = make_transaction_matrix(subset, VARIABLES)

    frequent_itemsets = apriori(
        matrix,
        min_support=MIN_SUPPORT,
        use_colnames=True,
        max_len=MAX_LEN,
    )

    if frequent_itemsets.empty:
        rules = pd.DataFrame()

    else:
        rules = association_rules(
            frequent_itemsets,
            metric="confidence",
            min_threshold=MIN_CONFIDENCE,
        )

        rules = rules.loc[
            rules["lift"] >= MIN_LIFT
        ].copy()

    if not rules.empty:

        # 동일 변수가 선행부와 후행부에 동시에 들어간 규칙 제외
        # 현재 one-hot 구조에서는 보통 발생하지 않지만 명시적으로 검증
        rules = rules.loc[
            rules.apply(
                lambda row:
                    get_variable_names(row["antecedents"])
                    .isdisjoint(
                        get_variable_names(row["consequents"])
                    ),
                axis=1,
            )
        ].copy()

        rules["cluster"] = cluster_id
        rules["cluster_name"] = CLUSTER_NAMES[cluster_id]
        rules["cluster_n"] = len(subset)

        rules["antecedent"] = rules["antecedents"].map(
            itemset_to_text
        )
        rules["consequent"] = rules["consequents"].map(
            itemset_to_text
        )

        rules["antecedent_length"] = rules["antecedents"].map(len)
        rules["consequent_length"] = rules["consequents"].map(len)

        rules["support_count"] = (
            rules["support"] * len(subset)
        ).round().astype(int)

        rules = rules.sort_values(
            ["lift", "confidence", "support"],
            ascending=[False, False, False],
        )

        all_rules.append(rules)

    summary_rows.append({
        "cluster": cluster_id,
        "cluster_name": CLUSTER_NAMES[cluster_id],
        "n": len(subset),
        "minimum_support": MIN_SUPPORT,
        "minimum_support_count": math.ceil(
            MIN_SUPPORT * len(subset)
        ),
        "minimum_confidence": MIN_CONFIDENCE,
        "minimum_lift": MIN_LIFT,
        "maximum_itemset_length": MAX_LEN,
        "frequent_itemset_count": len(frequent_itemsets),
        "rule_count": len(rules),
    })


# =========================================================
# 9. 결과 통합 및 저장
# =========================================================

OUTPUT_COLUMNS = [
    "cluster",
    "cluster_name",
    "cluster_n",
    "antecedent",
    "consequent",
    "antecedent_length",
    "consequent_length",
    "support_count",
    "support",
    "confidence",
    "lift",
    "antecedent support",
    "consequent support",
    "leverage",
    "conviction",
    "zhangs_metric",
]

if all_rules:
    result_df = pd.concat(all_rules, ignore_index=True)

    existing_output_columns = [
        col for col in OUTPUT_COLUMNS
        if col in result_df.columns
    ]

    result_df = result_df[existing_output_columns]

else:
    # 규칙이 0건이어도 오류 없이 헤더가 있는 파일 저장
    result_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

summary_df = pd.DataFrame(summary_rows)

rules_path = RESULT_DIR / "table5_association_rules_candidate.csv"
summary_path = RESULT_DIR / "association_rule_summary.csv"

result_df.to_csv(
    rules_path,
    index=False,
    encoding="utf-8-sig",
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)


# =========================================================
# 10. 최종 실행 결과 출력
# =========================================================

print("\n[최종 데이터 검증]")
print("전체 관측치:", len(analysis_df))
print("군집 번호:", sorted(analysis_df["cluster"].unique()))
print("군집 규모:", actual_cluster_counts)
print("사용 변수:", VARIABLES)

print("\n[연관규칙 분석 기준]")
print("minimum support:", MIN_SUPPORT)
print("minimum confidence:", MIN_CONFIDENCE)
print("minimum lift:", MIN_LIFT)
print("maximum itemset length:", MAX_LEN)

print("\n[군집별 결과]")
print(
    summary_df[
        [
            "cluster",
            "n",
            "minimum_support_count",
            "frequent_itemset_count",
            "rule_count",
        ]
    ].to_string(index=False)
)

print("\n[저장 파일]")
print(rules_path)
print(summary_path)

if result_df.empty:
    print("\n조건을 충족하는 연관규칙이 없습니다.")
else:
    print("\n[상위 규칙 미리보기]")
    print(
        result_df[
            [
                "cluster",
                "antecedent",
                "consequent",
                "support_count",
                "support",
                "confidence",
                "lift",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

ModuleNotFoundError: No module named 'mlxtend'